In [ ]:
import pandas as pd

#os 3 anos permitem repetir os testes sobre diferentes versoes do schema.
anos = [2022, 2023, 2024]

#centralizar o caminho evita divergencias ao trocar o ano analisado.
def caminho(ano):
    return f"../data/raw/{ano}/MICRODADOS_CADASTRO_CURSOS_{ano}.CSV"

#ler apenas o cabecalho testa o schema sem carregar a tabela inteira.
colunas = {}
for ano in anos:
    cab = pd.read_csv(caminho(ano), sep=";", encoding="latin1", nrows=0)
    colunas[ano] = set(cab.columns)
    print(ano, len(colunas[ano]), "colunas")

2022 200 colunas
2023 202 colunas
2024 223 colunas


In [ ]:
print("Entram em 2023:", colunas[2023] - colunas[2022])
print("Saem em 2024:", colunas[2023] - colunas[2024])
#ordenar a diferença ajuda a comparar entradas entre schemas sem depender da ordem do conjunto.
print("Entram em 2024:", sorted(colunas[2024] - colunas[2023]))

Entram em 2023: {'IN_CONFESSIONAL', 'IN_COMUNITARIA'}
Saem em 2024: {'QT_MAT_RVETNICO', 'QT_ING_RVETNICO', 'QT_CONC_RVETNICO'}
Entram em 2024: ['QT_CONC_RVIDOSO', 'QT_CONC_RVINTERN', 'QT_CONC_RVMEDAL', 'QT_CONC_RVPOVT', 'QT_CONC_RVPPI', 'QT_CONC_RVQUILO', 'QT_CONC_RVREFU', 'QT_CONC_RVTRANS', 'QT_ING_RVIDOSO', 'QT_ING_RVINTERN', 'QT_ING_RVMEDAL', 'QT_ING_RVPOVT', 'QT_ING_RVPPI', 'QT_ING_RVQUILO', 'QT_ING_RVREFU', 'QT_ING_RVTRANS', 'QT_MAT_RVIDOSO', 'QT_MAT_RVINTERN', 'QT_MAT_RVMEDAL', 'QT_MAT_RVPOVT', 'QT_MAT_RVPPI', 'QT_MAT_RVQUILO', 'QT_MAT_RVREFU', 'QT_MAT_RVTRANS']


In [ ]:
#o mesmo ano pode apontar para cursos ou IES; separar o tipo evita caminhos ambíguos.
def caminho(ano, tipo):
    arquivos = {
        "cursos": f"MICRODADOS_CADASTRO_CURSOS_{ano}.CSV",
        "ies": f"MICRODADOS_ED_SUP_IES_{ano}.CSV",
    }
    return f"../data/raw/{ano}/{arquivos[tipo]}"

#validar o cabeçalho de IES identifica mudanças do schema sem carregar os registros.
colunas_ies = {}
for ano in anos:
    cab = pd.read_csv(caminho(ano, "ies"), sep=";", encoding="latin1", nrows=0)
    colunas_ies[ano] = set(cab.columns)
    print(ano, len(colunas_ies[ano]), "colunas")

print("Entram em 2023:", sorted(colunas_ies[2023] - colunas_ies[2022]))
print("Saem/entram 2024:", sorted(colunas_ies[2023] ^ colunas_ies[2024]))

2022 81 colunas
2023 84 colunas
2024 84 colunas
Entram em 2023: ['IN_COMUNITARIA', 'IN_CONFESSIONAL', 'TP_REDE']
Saem/entram 2024: []


In [ ]:
#buscar por um termo permite descobrir campos relacionados sem conhecer todos os nomes antecipadamente.
def achar(palavra):
    return sorted(c for c in colunas[2024] if palavra in c)

for p in ["REGIAO", "CINE", "FIES", "PROUNI", "PRETA", "PARDA", "INDIGENA", "RESERVA"]:
    print(p, achar(p))

REGIAO ['CO_REGIAO', 'NO_REGIAO']
CINE ['CO_CINE_AREA_DETALHADA', 'CO_CINE_AREA_ESPECIFICA', 'CO_CINE_AREA_GERAL', 'CO_CINE_ROTULO', 'NO_CINE_AREA_DETALHADA', 'NO_CINE_AREA_ESPECIFICA', 'NO_CINE_AREA_GERAL', 'NO_CINE_ROTULO']
FIES ['QT_CONC_FIES', 'QT_CONC_NRPFIES', 'QT_CONC_RPFIES', 'QT_ING_FIES', 'QT_ING_NRPFIES', 'QT_ING_RPFIES', 'QT_MAT_FIES', 'QT_MAT_NRPFIES', 'QT_MAT_RPFIES']
PROUNI ['QT_CONC_PROUNII', 'QT_CONC_PROUNIP', 'QT_ING_PROUNII', 'QT_ING_PROUNIP', 'QT_MAT_PROUNII', 'QT_MAT_PROUNIP']
PRETA ['QT_CONC_PRETA', 'QT_ING_PRETA', 'QT_MAT_PRETA']
PARDA ['QT_CONC_PARDA', 'QT_ING_PARDA', 'QT_MAT_PARDA']
INDIGENA ['QT_CONC_INDIGENA', 'QT_ING_INDIGENA', 'QT_MAT_INDIGENA']
RESERVA ['QT_CONC_RESERVA_VAGA', 'QT_ING_RESERVA_VAGA', 'QT_MAT_RESERVA_VAGA']


In [8]:
for p in ["MUNICIPIO", "UF", "CAPITAL", "GRATUITO", "NO_IES", "SG_IES",
          "NO_CURSO", "MANTENEDORA", "VG_TOTAL", "INSCRITO_TOTAL"]:
    print(p, achar(p))

MUNICIPIO ['CO_MUNICIPIO', 'NO_MUNICIPIO']
UF ['CO_UF', 'NO_UF', 'SG_UF']
CAPITAL ['IN_CAPITAL']
GRATUITO ['IN_GRATUITO']
NO_IES []
SG_IES []
NO_CURSO ['NO_CURSO']
MANTENEDORA []
VG_TOTAL ['QT_VG_TOTAL', 'QT_VG_TOTAL_DIURNO', 'QT_VG_TOTAL_EAD', 'QT_VG_TOTAL_NOTURNO']
INSCRITO_TOTAL ['QT_INSCRITO_TOTAL', 'QT_INSCRITO_TOTAL_DIURNO', 'QT_INSCRITO_TOTAL_EAD', 'QT_INSCRITO_TOTAL_NOTURNO']


In [ ]:
#essa lista fixa define as variáveis que devem existir para o modelo integrado funcionar.
escolhidas = [
    "NU_ANO_CENSO", "CO_CURSO", "CO_MUNICIPIO", "CO_IES", "CO_CINE_ROTULO",
    "QT_ING", "QT_MAT", "QT_CONC", "QT_ING_FIES",
    "QT_ING_PROUNII", "QT_ING_PROUNIP",
    "QT_ING_PRETA", "QT_ING_PARDA", "QT_ING_INDIGENA", "QT_ING_RESERVA_VAGA",
    "TP_GRAU_ACADEMICO", "TP_NIVEL_ACADEMICO", "TP_MODALIDADE_ENSINO",
    "TP_DIMENSAO", "TP_REDE", "TP_CATEGORIA_ADMINISTRATIVA",
    "TP_ORGANIZACAO_ACADEMICA", "CO_REGIAO", "NO_REGIAO",
]
#para cada campo obrigatório, listar apenas os anos sem a coluna mostra quais incompatibilidades existem.
for c in escolhidas:
    faltando = [a for a in anos if c not in colunas[a]]
    print(c, "OK" if not faltando else f"NÃO existe em {faltando}")

NU_ANO_CENSO OK
CO_CURSO OK
CO_MUNICIPIO OK
CO_IES OK
CO_CINE_ROTULO OK
QT_ING OK
QT_MAT OK
QT_CONC OK
QT_ING_FIES OK
QT_ING_PROUNII OK
QT_ING_PROUNIP OK
QT_ING_PRETA OK
QT_ING_PARDA OK
QT_ING_INDIGENA OK
QT_ING_RESERVA_VAGA OK
TP_GRAU_ACADEMICO OK
TP_NIVEL_ACADEMICO OK
TP_MODALIDADE_ENSINO OK
TP_DIMENSAO OK
TP_REDE OK
TP_CATEGORIA_ADMINISTRATIVA OK
TP_ORGANIZACAO_ACADEMICA OK
CO_REGIAO OK
NO_REGIAO OK


In [ ]:
#2024 mudou a representaçao da reserva, o mapa preserva a origem correta dos dados em cada ano.
cols = {
    2022: ["QT_ING", "QT_ING_RVETNICO", "QT_ING_PRETA", "QT_ING_PARDA", "QT_ING_INDIGENA"],
    2023: ["QT_ING", "QT_ING_RVETNICO", "QT_ING_PRETA", "QT_ING_PARDA", "QT_ING_INDIGENA"],
    2024: ["QT_ING", "QT_ING_RVPPI", "QT_ING_RVQUILO", "QT_ING_RVPOVT",
           "QT_ING_PRETA", "QT_ING_PARDA", "QT_ING_INDIGENA"],
}

for ano in anos:
    d = pd.read_csv(caminho(ano, "cursos"), sep=";", encoding="latin1", usecols=cols[ano])
    total = d["QT_ING"].sum()

#antes de 2024, a reserva etnica era agregada em uma coluna. em 2024 as categorias sao somadas separadamente.
    if ano < 2024:
        reserva = d["QT_ING_RVETNICO"].sum()
    else:
        reserva = d[["QT_ING_RVPPI", "QT_ING_RVQUILO", "QT_ING_RVPOVT"]].sum().sum()
        print("   2024 por grupo:", d[["QT_ING_RVPPI", "QT_ING_RVQUILO", "QT_ING_RVPOVT"]].sum().to_dict())

    ppi_cor = d[["QT_ING_PRETA", "QT_ING_PARDA", "QT_ING_INDIGENA"]].sum().sum()

    print(f"{ano}: reserva étnica = {reserva/total*100:.2f}% dos ingressantes | "
          f"pretos+pardos+indígenas = {ppi_cor/total*100:.2f}%")

2022: reserva étnica = 1.34% dos ingressantes | pretos+pardos+indígenas = 34.10%
2023: reserva étnica = 1.45% dos ingressantes | pretos+pardos+indígenas = 35.39%
   2024 por grupo: {'QT_ING_RVPPI': 79969, 'QT_ING_RVQUILO': 2314, 'QT_ING_RVPOVT': 758}
2024: reserva étnica = 1.66% dos ingressantes | pretos+pardos+indígenas = 36.02%


In [11]:
def achar_ies(palavra):
    return sorted(c for c in colunas_ies[2024] if palavra in c)

for p in ["NO_IES", "SG_IES", "MANTENEDORA", "UF", "MUNICIPIO", "TP_", "IN_"]:
    print(p, achar_ies(p))

NO_IES ['NO_IES']
SG_IES ['SG_IES']
MANTENEDORA ['CO_MANTENEDORA', 'NO_MANTENEDORA']
UF ['CO_UF_IES', 'NO_UF_IES', 'SG_UF_IES']
MUNICIPIO ['CO_MUNICIPIO_IES', 'NO_MUNICIPIO_IES']
TP_ ['TP_CATEGORIA_ADMINISTRATIVA', 'TP_ORGANIZACAO_ACADEMICA', 'TP_REDE']
IN_ ['IN_ACESSO_OUTRAS_BASES', 'IN_ACESSO_PORTAL_CAPES', 'IN_ASSINA_OUTRA_BASE', 'IN_BUSCA_INTEGRADA', 'IN_CAPITAL_IES', 'IN_CATALOGO_ONLINE', 'IN_COMUNITARIA', 'IN_CONFESSIONAL', 'IN_PARTICIPA_REDE_SOCIAL', 'IN_REPOSITORIO_INSTITUCIONAL', 'IN_SERVICO_INTERNET']


In [ ]:
#a tabela cruzada testa se a dimensao está associada a ausencia de municipio, como esperado para cursos ead.
d = pd.read_csv(caminho(2023, "cursos"), sep=";", encoding="latin1",
                usecols=["CO_MUNICIPIO", "CO_REGIAO", "TP_DIMENSAO"])
print(pd.crosstab(d["TP_DIMENSAO"], d["CO_MUNICIPIO"].isna()))
print("Região nula:", d["CO_REGIAO"].isna().sum())

CO_MUNICIPIO   False  True 
TP_DIMENSAO                
1              35739      0
2             624832      4
3                  0  10576
4                  0    459
Região nula: 11039


In [13]:
d = pd.read_csv(caminho(2023, "cursos"), sep=";", encoding="latin1",
                usecols=["TP_DIMENSAO", "QT_ING", "QT_MAT"])
print(d.groupby("TP_DIMENSAO")[["QT_ING", "QT_MAT"]].sum())

              QT_ING   QT_MAT
TP_DIMENSAO                  
1            1679790  5063936
2            3312712  4910515
3                  0        0
4               1690     2766


In [14]:
d = pd.read_csv(caminho(2022, "cursos"), sep=";", encoding="latin1",
                usecols=["TP_CATEGORIA_ADMINISTRATIVA", "TP_REDE"])
print(pd.crosstab(d["TP_CATEGORIA_ADMINISTRATIVA"], d["TP_REDE"]))

TP_REDE                         1       2
TP_CATEGORIA_ADMINISTRATIVA              
1                            9781       0
2                            7485       0
3                            1001       0
4                               0  493177
5                               0   61226
7                             349       0


In [ ]:
#testar a chave primeiro sem e depois com dimensao separa duplicidades reais de registros válidos em dimensoes diferentes.
usar = ["CO_CURSO", "CO_MUNICIPIO", "TP_DIMENSAO", "TP_MODALIDADE_ENSINO"]
for ano in anos:
    d = pd.read_csv(caminho(ano, "cursos"), sep=";", encoding="latin1", usecols=usar)
    dup_a = d.duplicated(subset=["CO_CURSO", "CO_MUNICIPIO"], keep=False).sum()
    dup_b = d.duplicated(subset=["CO_CURSO", "CO_MUNICIPIO", "TP_DIMENSAO"], keep=False).sum()
    var = (d.groupby("CO_CURSO")["TP_DIMENSAO"].nunique() > 1).sum()
    print(ano, "linhas duplicadas sem/com TP_DIMENSAO:", dup_a, dup_b,
          "| cursos com mais de uma TP_DIMENSAO:", var)

2022 linhas duplicadas sem/com TP_DIMENSAO: 792 0 | cursos com mais de uma TP_DIMENSAO: 9154
2023 linhas duplicadas sem/com TP_DIMENSAO: 922 0 | cursos com mais de uma TP_DIMENSAO: 10507
2024 linhas duplicadas sem/com TP_DIMENSAO: 904 0 | cursos com mais de uma TP_DIMENSAO: 11248


In [ ]:
#agrupar por CO_CURSO verifica quais atributos variam dentro da identidade do curso ao longo dos três anos.
usar = ["CO_CURSO", "CO_IES", "CO_CINE_ROTULO", "NO_CURSO", "TP_GRAU_ACADEMICO",
        "TP_NIVEL_ACADEMICO", "TP_MODALIDADE_ENSINO", "IN_GRATUITO",
        "TP_DIMENSAO", "QT_VG_TOTAL", "QT_INSCRITO_TOTAL"]

for ano in anos:
    d = pd.read_csv(caminho(ano, "cursos"), sep=";", encoding="latin1", usecols=usar)
    print(f"\n=== {ano} ===")
    for c in ["CO_IES", "CO_CINE_ROTULO", "NO_CURSO", "TP_GRAU_ACADEMICO",
              "TP_NIVEL_ACADEMICO", "TP_MODALIDADE_ENSINO", "IN_GRATUITO"]:
        n = (d.groupby("CO_CURSO")[c].nunique(dropna=False) > 1).sum()
        print(f"{c}: {n} cursos com mais de um valor")
    print(pd.crosstab(d["TP_DIMENSAO"], d["TP_MODALIDADE_ENSINO"]))
    print(d.groupby("TP_DIMENSAO")[["QT_VG_TOTAL", "QT_INSCRITO_TOTAL"]].sum())

#regra de rede nos anos em que as duas colunas existem no arquivo de IES
for ano in [2023, 2024]:
    i = pd.read_csv(caminho(ano, "ies"), sep=";", encoding="latin1",
                    usecols=["TP_CATEGORIA_ADMINISTRATIVA", "TP_REDE"])
    print(f"\nIES {ano}")
    print(pd.crosstab(i["TP_CATEGORIA_ADMINISTRATIVA"], i["TP_REDE"]))


=== 2022 ===
CO_IES: 0 cursos com mais de um valor
CO_CINE_ROTULO: 0 cursos com mais de um valor
NO_CURSO: 0 cursos com mais de um valor
TP_GRAU_ACADEMICO: 0 cursos com mais de um valor
TP_NIVEL_ACADEMICO: 0 cursos com mais de um valor
TP_MODALIDADE_ENSINO: 0 cursos com mais de um valor
IN_GRATUITO: 0 cursos com mais de um valor
TP_MODALIDADE_ENSINO      1       2
TP_DIMENSAO                        
1                     36089       0
2                         0  527328
3                         0    9207
4                         0     395
             QT_VG_TOTAL  QT_INSCRITO_TOTAL
TP_DIMENSAO                                
1                5658590            8811102
2                      0                  0
3               17171895            8176586
4                      0                  0

=== 2023 ===
CO_IES: 0 cursos com mais de um valor
CO_CINE_ROTULO: 0 cursos com mais de um valor
NO_CURSO: 0 cursos com mais de um valor
TP_GRAU_ACADEMICO: 0 cursos com mais de um valor
T

In [ ]:
# 1 - em 2022, que municipio tem as linhas de dimensao 3 e 4?
#filtrar essas dimensoes permite investigar seus registros anomalos antes de definir qualquer tratamento.
d = pd.read_csv(caminho(2022, "cursos"), sep=";", encoding="latin1",
                usecols=["TP_DIMENSAO", "NO_MUNICIPIO", "QT_ING", "QT_MAT"])
x = d[d["TP_DIMENSAO"].isin([3, 4])]
print(x.groupby("TP_DIMENSAO")[["QT_ING", "QT_MAT"]].sum())
print(x.groupby("TP_DIMENSAO")["NO_MUNICIPIO"]
        .agg(lambda s: s.value_counts().head(3).to_dict()))

#atributos de cursos e IES mudam entre anos?
#juntar os anos por chave e contar valores distintos mede as mudanças temporais que podem afetar integrações.
def mudancas(tipo, chave, cols):
    partes = []
    for ano in anos:
        p = pd.read_csv(caminho(ano, tipo), sep=";", encoding="latin1",
                        usecols=[chave] + cols)
        partes.append(p.drop_duplicates(subset=[chave]).assign(ano=ano))
    t = pd.concat(partes)
    print(f"\n{tipo}: {t[chave].nunique()} {chave} distintos")
    for c in cols:
        n = (t.groupby(chave)[c].nunique(dropna=False) > 1).sum()
        print(f"  {c}: {n} mudam de valor entre anos")

mudancas("cursos", "CO_CURSO",
         ["NO_CURSO", "TP_GRAU_ACADEMICO", "TP_NIVEL_ACADEMICO",
          "TP_MODALIDADE_ENSINO", "IN_GRATUITO", "CO_IES", "CO_CINE_ROTULO"])
mudancas("ies", "CO_IES",
         ["NO_IES", "SG_IES", "TP_ORGANIZACAO_ACADEMICA",
          "TP_CATEGORIA_ADMINISTRATIVA", "CO_MANTENEDORA", "NO_MANTENEDORA"])

/tmp/ipykernel_26374/893849879.py:2: DtypeWarning: Columns (6: NO_MUNICIPIO) have mixed types. Specify dtype option on import or set low_memory=False.
  d = pd.read_csv(caminho(2022, "cursos"), sep=";", encoding="latin1",


             QT_ING  QT_MAT
TP_DIMENSAO                
3                 0       0
4              1788    2818
TP_DIMENSAO
3    {}
4    {}
Name: NO_MUNICIPIO, dtype: object

cursos: 50577 CO_CURSO distintos
  NO_CURSO: 265 mudam de valor entre anos
  TP_GRAU_ACADEMICO: 335 mudam de valor entre anos
  TP_NIVEL_ACADEMICO: 0 mudam de valor entre anos
  TP_MODALIDADE_ENSINO: 0 mudam de valor entre anos
  IN_GRATUITO: 53 mudam de valor entre anos
  CO_IES: 346 mudam de valor entre anos
  CO_CINE_ROTULO: 2746 mudam de valor entre anos

ies: 2792 CO_IES distintos
  NO_IES: 171 mudam de valor entre anos
  SG_IES: 153 mudam de valor entre anos
  TP_ORGANIZACAO_ACADEMICA: 37 mudam de valor entre anos
  TP_CATEGORIA_ADMINISTRATIVA: 38 mudam de valor entre anos
  CO_MANTENEDORA: 121 mudam de valor entre anos
  NO_MANTENEDORA: 272 mudam de valor entre anos


In [ ]:
#ies que mudaram de rede, quantas matriculas representam?
#a mudança e agregada por ies e dividida pelo total anual para testar se a discrepancia e relevante.
partes = []
for ano in anos:
    d = pd.read_csv(caminho(ano, "cursos"), sep=";", encoding="latin1",
                    usecols=["CO_IES", "TP_REDE", "QT_MAT"], low_memory=False)
    g = d.groupby("CO_IES").agg(rede=("TP_REDE", "first"), mat=("QT_MAT", "sum")).reset_index()
    partes.append(g.assign(ano=ano))
t = pd.concat(partes)
mudou = t.groupby("CO_IES")["rede"].nunique() > 1
ids = mudou[mudou].index
print(len(ids), "IES mudaram de rede")
tot = t.groupby("ano")["mat"].sum()
print((t[t["CO_IES"].isin(ids)].groupby("ano")["mat"].sum() / tot * 100).round(2), "% das matrículas")

#CINE. o codigo muda por reclassificaçao ou os cursos mudaram de área?
# os testes separam alteração da area geral, diversidade de rotulos e mudança de descriçao do mesmo codigo.
partes = []
for ano in anos:
    d = pd.read_csv(caminho(ano, "cursos"), sep=";", encoding="latin1",
                    usecols=["CO_CURSO", "CO_CINE_AREA_GERAL", "CO_CINE_ROTULO",
                             "NO_CINE_ROTULO", "QT_ING"], low_memory=False)
    partes.append(d.assign(ano=ano))
c = pd.concat(partes)
geral = c.groupby("CO_CURSO")["CO_CINE_AREA_GERAL"].nunique() > 1
print(geral.sum(), "cursos mudaram de área GERAL")
print((c.groupby("CO_CINE_ROTULO")["NO_CINE_ROTULO"].nunique() > 1).sum(),
      "códigos de rótulo com mais de um nome")
print(c.groupby("ano")["CO_CINE_ROTULO"].nunique(), "rótulos distintos por ano")

# 2) 2022: que código de município têm as linhas de dimensão 3 e 4?
# Usar códigos geográficos identifica valores sentinela que não deveriam ser tratados como municípios reais.
d = pd.read_csv(caminho(2022, "cursos"), sep=";", encoding="latin1",
                usecols=["TP_DIMENSAO", "CO_MUNICIPIO", "CO_UF", "CO_REGIAO"], low_memory=False)
x = d[d["TP_DIMENSAO"].isin([3, 4])]
print(x.groupby("TP_DIMENSAO")[["CO_MUNICIPIO", "CO_UF", "CO_REGIAO"]]
        .agg(lambda s: s.value_counts(dropna=False).head(3).to_dict()))

1 IES mudaram de rede
ano
2022    0.01
2023    0.01
2024    0.01
Name: mat, dtype: float64 % das matrículas
219 cursos mudaram de área GERAL
1 códigos de rótulo com mais de um nome
ano
2022    351
2023    354
2024    353
Name: CO_CINE_ROTULO, dtype: int64 rótulos distintos por ano
            CO_MUNICIPIO      CO_UF  CO_REGIAO
TP_DIMENSAO                                   
3              {0: 9207}  {0: 9207}  {0: 9207}
4               {0: 395}   {0: 395}   {0: 395}


In [ ]:
#em 2022 o municipio 0 so aparece nas dimensões 3 e 4?
#essa tabela confirma se o valor sentinela 0 e exclusivo das dimensoes investigadas.
d = pd.read_csv(caminho(2022, "cursos"), sep=";", encoding="latin1",
                usecols=["TP_DIMENSAO", "CO_MUNICIPIO"], low_memory=False)
print(pd.crosstab(d["TP_DIMENSAO"], d["CO_MUNICIPIO"] == 0))

#o rotulo CINE que tem dois nomes (reusa o dataframe c do teste anterior)
n = c.groupby("CO_CINE_ROTULO")["NO_CINE_ROTULO"].nunique()
cod = n[n > 1].index[0]
print(c[c["CO_CINE_ROTULO"] == cod].groupby(["ano", "NO_CINE_ROTULO"]).size())

CO_MUNICIPIO   False  True 
TP_DIMENSAO                
1              36089      0
2             527327      1
3                  0   9207
4                  0    395
ano   NO_CINE_ROTULO                           
2022  Engenharia de computação (DCN Engenharia)    2190
2023  Engenharia de computação (DCN Engenharia)    2452
2024  Engenharia de computação                     3731
dtype: int64
